In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame({'Name':['Prabin','Samir','Chadup','Bikesh','Deepesh','Aditi','Alwin'],'Age':[21,23,34,18,21,23,34], 'Ed_lvl':['PhD','Bachelors','High School','Masters','High School','Bachelors','Masters']})
df

,Name,Age,Ed_lvl
0,Prabin,21,PhD
1,Samir,23,Bachelors
2,Chadup,34,High School
3,Bikesh,18,Masters
4,Deepesh,21,High School
5,Aditi,23,Bachelors
6,Alwin,34,Masters


In [3]:
# since the normal hirearchy is 'High School' < 'Bachelors' < Masters' < 'PhD'.
#  the generic encoding would follow:
ed_lvl_encoding = {'High School':1,'Bachelors':2,'Masters':3,'PhD':4,'NA':0} # NA is added for unknown cataegories that may be encountered in the data

In [4]:
# ed_lvl_encoding_alphabetically 
encoding_alphabetically =  {'High School':2,'Bachelors':1,'Masters':3,'PhD':4,'NA':0} # NA is still kept at 0 .

In [5]:
class ordinalencoder():
    def __init__(self,encoding = {}):
        self.encoding = encoding
    def encode_column(self,column):
        self.column = column
        self.encoded_column = []
        for itr in self.column.values:
            if ( itr in self.encoding.keys()):
                itr = self.encoding[itr]
            else:
                itr = self.encoding['NA']
            self.encoded_column.append(itr)
        return self.encoded_column

In [6]:
oe_general = ordinalencoder(ed_lvl_encoding)
df['Ed_lvl_encoded'] = oe_general.encode_column(df['Ed_lvl'])
df

,Name,Age,Ed_lvl,Ed_lvl_encoded
0,Prabin,21,PhD,4
1,Samir,23,Bachelors,2
2,Chadup,34,High School,1
3,Bikesh,18,Masters,3
4,Deepesh,21,High School,1
5,Aditi,23,Bachelors,2
6,Alwin,34,Masters,3


In [ ]:
oe_alpha = ordinalencoder(encoding_alphabetically)
df['Ed_lvl_encoded_aplha'] = oe_alpha.encode_column(df['Ed_lvl'])
df
# so it seems, encoding ed_lvl alphabetically results rank of  'Bachelors' and 'High school' interchanged.

,Name,Age,Ed_lvl,Ed_lvl_encoded,Ed_lvl_encoded_aplha
0,Prabin,21,PhD,4,4
1,Samir,23,Bachelors,2,1
2,Chadup,34,High School,1,2
3,Bikesh,18,Masters,3,3
4,Deepesh,21,High School,1,2
5,Aditi,23,Bachelors,2,1
6,Alwin,34,Masters,3,3


In [ ]:
# Q. if a linear model later reads "higher encoded value = more education," what specifically 
# breaks in its coefficients if the alphabetical mapping had been used instead ?

# answer:    under such scenario, the intended or factual order of categories is disrupted. the encoded values lose its 
#            significance , as then the encoded values dont reflect the natural or intended hirearchy of categories.
#            example: As presented above, alphabetical encoding  may cause encoded value of 'Bachelors' to be '1' which is less than 
#            encoded value of '2' for 'High School'. this is  a false representation where order is a priority. and can lead linear models to poor predictions.

#            Suppose the real underlying relationship between education and, say, salary is genuinely monotonic and roughly linear: 
#            High School → 30k, Bachelors → 50k, Masters → 70k, PhD → 90k. Under the true hierarchy encoding (1,2,3,4), those four points 
#            sit almost perfectly on a line — a linear model fits a clean, honest slope.
#            but when we do it with a alphabetical encoding instead, 'bachelors' is now x = 1 and 'High school' is now x = 2,
#            the four points are not linear anymore. the point for 'bachelors' with (x=1,salary= 50k) ranks higher than the point for 'High school' with (x = 2 ,salary = 30k).
#            there is now a dip in the graph, a single line cannot pass through all the points anymore.
#            as a result, the fitted line must pass through points or close to all the points which attains least sum of residuals to accomodate the contradiction.
#            but still contradiction can only attempted to be reduced,it is not removed completely.  
#            also the overall coefficient of the model is also changed and line does shift in y-axis.

#           The two b1 values are the heart of this whole question, and they land beautifully: 20 vs. 16.
#   That directly falsifies what you guessed earlier ("the overall coefficient of the model is not changed")
#   — it does change, by a full 4 units, purely because two category labels swapped positions on the x-axis.
#     Nothing about the real world changed; only your arbitrary ordering choice did, and the model's learned slope 
#   moved as a direct consequence.

In [26]:
import numpy as np
x_true = [1, 2, 3, 4]
x_alpha = [1, 2, 3, 4]  # same x positions, but which category sits where has changed
y_true  = [30, 50, 70, 90]   # HS, Bach, Mast, PhD salary — matched to x_true order
y_alpha = [50, 30, 70, 90]   # Bach, HS, Mast, PhD — matched to x_alpha order (the swap)

b1_true = np.cov(x_true, y_true, bias=True)[0,1] / np.var(x_true)
print('b1_true',b1_true)
b2_alpha = np.cov(x_alpha, y_alpha, bias=True)[0,1] / np.var(x_alpha)
print('b2_alpha',b2_alpha)

b1_true 20.0
b2_alpha 16.0


In [29]:
y_mean = sum(y_true)/len(y_true)
print('y_mean',y_mean)
x_mean = sum(x_true)/len(x_true)
print('x_mean',x_mean)


y_mean 60.0
x_mean 2.5


In [30]:
16 * 2.5

40.0